# Wczytanie Danych | Feature Engineering | UMAP 

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
import umap.umap_ as umap 
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("="*80)
print(">>> KROK 1: ŁADOWANIE DANYCH, INŻYNIERIA CECH I REDUKCJA UMAP <<<")
print("="*80)

# ==========================================
# 1. ŁADOWANIE DANYCH 
# ==========================================

ALL_FILES_PATTERN = "../dane/treningowe/*_netflow-extended.csv" 
RANDOM_SEED = 42

def load_stratified_data(file_pattern, sample_size=10000):
    files = glob.glob(file_pattern)
    dfs = []
    print(f"[*] Pobieranie próbek z {len(files)} plików...")
    for path in files:
        try:
            df_temp = pd.read_csv(path, on_bad_lines="skip", low_memory=False)
            df_sampled = df_temp.sample(n=sample_size, random_state=RANDOM_SEED) if len(df_temp) > sample_size else df_temp
            df_sampled['original_file'] = os.path.basename(path)
            dfs.append(df_sampled)
        except Exception as e: 
            print(f"  [ERR] {path}: {e}")
            
    if not dfs: return pd.DataFrame()
    final_df = pd.concat(dfs, ignore_index=True).dropna(axis=1, how='all')
    if 'StartTime' in final_df.columns: 
        final_df.sort_values(by='StartTime', inplace=True)
    return final_df

df_train = load_stratified_data("../dane/treningowe/*geo-1*.csv", sample_size=20000)
df_test = load_stratified_data("../dane/testowe/*geo-1*.csv", sample_size=20000)

common_cols = df_train.columns.intersection(df_test.columns)
df_train, df_test = df_train[common_cols].copy(), df_test[common_cols].copy()

# ==========================================
# 2. INŻYNIERIA CECH 
# ==========================================
NUM_COLS = ['Dur', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 
            'Bytes_per_Pkt', 'Pkts_Ratio', 'Pkts_Freq', 'Bytes_Ratio'] 

BIN_COLS = ['argus_src_loss', 'argus_dst_loss', 'argus_encap', 'argus_mismatch',
            'is_well_known_port', 'is_ephemeral_port'] 

TARGET_PORTS = [
    21, 22, 23, 3389, 5900, 80, 443, 8080, 8081, 8888, 3128, 1080,
    25, 53, 110, 143, 1433, 3306, 5432, 6379, 27017,
    445, 139, 2323, 5060, 5061, 500, 4500, 123, 1900, 5353, 5683
]

def clean_and_engineer_train(df, valid_states, valid_protos):
    df = df.copy()
    
    for c in ['Sport', 'Dport', 'TotBytes', 'TotPkts', 'SrcPkts', 'DstPkts', 'SrcBytes', 'DstBytes', 'Dur']:
        if c in df.columns: 
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
            
    df['Bytes_per_Pkt'] = df['TotBytes'] / (df['TotPkts'] + 1e-6)
    df['Pkts_Ratio'] = df['SrcPkts'] / (df['DstPkts'] + 1e-6)
    df['Pkts_Freq'] = df['TotPkts'] / (df['Dur'] + 1e-6)
    
    # 1. Asymetria ruchu (Kluczowe dla oddzielenia skanów od połączeń)
    df['Bytes_Ratio'] = df['SrcBytes'] / (df['DstBytes'] + 1e-6)
    
    # 2. Inteligentna obsługa "długiego ogona" portów
    df['is_well_known_port'] = ((df['Dport'] > 0) & (df['Dport'] <= 1024)).astype(int)
    df['is_ephemeral_port'] = (df['Dport'] >= 49152).astype(int)

    # --- TWARDE KODOWANIE (ORTOGONALNE) ---
    for p in TARGET_PORTS: 
        df[f'is_dport_{p}'] = (df['Dport'] == p).astype(int)
        
    # Flagi Argusa
    if 'Flgs' in df.columns:
        flgs = df['Flgs'].astype(str).str.lower()
        df['argus_src_loss'] = flgs.apply(lambda x: 1 if 's' in x else 0)
        df['argus_dst_loss'] = flgs.apply(lambda x: 1 if 'd' in x else 0)
        df['argus_encap']    = flgs.apply(lambda x: 1 if 'e' in x else 0)
        df['argus_mismatch'] = flgs.apply(lambda x: 1 if '*' in x else 0)
    else:
        for c in BIN_COLS: df[c] = 0
            
    # Kategoryzacja stanu i protokołu
    if 'State' in df.columns:
        df['State'] = df['State'].astype(str).str.upper()
        df.loc[~df['State'].isin(valid_states), 'State'] = 'OTHER'
            
    if 'Proto' in df.columns:
        df['Proto'] = df['Proto'].astype(str).str.lower()
        df.loc[~df['Proto'].isin(valid_protos), 'Proto'] = 'OTHER'

    return df

print("[*] Generowanie profili behawioralnych...")
top_states = df_train['State'].astype(str).str.upper().value_counts().nlargest(20).index.tolist()
top_protos = df_train['Proto'].astype(str).str.lower().value_counts().nlargest(10).index.tolist()

df_train_eng = clean_and_engineer_train(df_train, top_states, top_protos)

# Logarytmowanie cech silnie skośnych (wyciszanie zjawisk wolumetrycznych)
valid_num = [c for c in NUM_COLS if c in df_train_eng.columns]
for col in valid_num: 
    df_train_eng[col] = np.log1p(df_train_eng[col].clip(lower=0))

current_bin_cols = [c for c in df_train_eng.columns if c.startswith('is_dport_') or c in BIN_COLS]

# ==========================================
# 3. TRANSFORMACJA I UMAP
# ==========================================
print("[*] Skalowanie (RobustScaler) oraz kodowanie (OHE)...")
transformer = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), valid_num),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['State', 'Proto']),
        ('bin', 'passthrough', current_bin_cols) 
    ], remainder='drop'
)

X_train_matrix = transformer.fit_transform(df_train_eng)

if np.isnan(X_train_matrix).any() or np.isinf(X_train_matrix).any():
    X_train_matrix = np.nan_to_num(X_train_matrix)

print("[*] Redukcja wymiarowości (UMAP)...")
reducer = umap.UMAP(
    n_components=10,       
    n_neighbors=15, 
    min_dist=0.0,         
    metric='euclidean', 
    n_jobs=1, 
    init='random',        # Wymagane przy dyskretnych sieciach (unika błędu Eigengap wynikającego z spectral)
    random_state=42
)

X_train_umap = reducer.fit_transform(X_train_matrix)

models_dir = '../modele'
if not os.path.exists(models_dir): 
    os.makedirs(models_dir)
    
joblib.dump(reducer, os.path.join(models_dir, 'umap_reducer.pkl'))
print("[+] Zapisano wyuczony model UMAP na dysku.")


print(f"[+] UMAP zakończony sukcesem. Kształt macierzy: {X_train_umap.shape}")
print("[+] Dane są gotowe do ekstrakcji wiedzy.")

>>> KROK 1: ŁADOWANIE DANYCH, INŻYNIERIA CECH I REDUKCJA UMAP <<<
[*] Pobieranie próbek z 7 plików...
[*] Pobieranie próbek z 1 plików...
[*] Generowanie profili behawioralnych...
[*] Skalowanie (RobustScaler) oraz kodowanie (OHE)...
[*] Redukcja wymiarowości (UMAP)...
[+] Zapisano wyuczony model UMAP na dysku.
[+] UMAP zakończony sukcesem. Kształt macierzy: (65765, 10)
[+] Dane są gotowe do ekstrakcji wiedzy.


## Wyzwania związane z analizą surowego ruchu sieciowego (NetFlow/Argus) wymagają wysoce specjalistycznego podejścia do przygotowania danych. Powyższy kod realizuje trzy krytyczne kroki transformacji:
### 1.Celowy dobór i inżynieria cech (Feature Engineering)
Zamiast opierać się wyłącznie na surowych licznikach, wygenerowano cechy behawioralne, które oddają "intencję" połączenia:
- <span style="color: red;"> Bytes_per_Pkt </span> (Rozmiar ładunku): Kluczowa metryka pozwalająca odróżnić puste pakiety skanujące (np. L4 SYN Scan o rozmiarze ok. 60 bajtów) od ataków aplikacyjnych przenoszących payload (np. brute-force, eksploitacje).
- <span style="color: red;"> Pkts_Ratio </span> (Asymetria ruchu): Stosunek pakietów wysłanych do odebranych doskonale izoluje ruch jednostronny (np. spoofing, backscatter, skanowanie UDP) od pełnych, dwustronnych wymian TCP.
- Selektywne mapowanie portów <span style="color: red;"> is_dport_X </span>: Zamiast traktować port docelowy jako ciągłą zmienną numeryczną (co jest błędem matematycznym, gdyż port 80 nie jest "mniejszy" od portu 443 w sensie wartości), zastosowano kodowanie binarne (One-Hot) wyłącznie dla krytycznych portów infrastrukturalnych i wektorów ataków (np. SMB, SIP, DNS). Zapobiega to przeuczeniu modelu (overfitting) na tysiącach losowych portów efemerycznych.
### 2. Przekształcenia rozkładów (Log-Transform i RobustScaler)
Dane sieciowe z natury charakteryzują się ekstremalnymi rozkładami grubooogonowymi (np. jeden atak DDoS może wygenerować milion pakietów, podczas gdy skan zaledwie dwa).
- Zastosowanie transformacji logarytmicznej <span style="color: red;"> np.log1p() </span> kompresuje te ekstremalne wartości, "zbliżając" je do rozkładu normalnego.
- Następnie użyto algorytmu <span style="color: red;"> RobustScaler </span>, który w przeciwieństwie do standardowego skalowania (StandardScaler) jest odporny na gigantyczne wartości odstające (outliery), opierając się na medianie i rozstępie kwartylnym.
### 3. Pokonanie "Klątwy Wymiarowości" (UMAP do 5 wymiarów)
Po procesie inżynierii i kodowania kategorycznego (One-Hot Encoding dla stanów i protokołów), szerokość naszego zbioru danych drastycznie wzrosła. Użycie algorytmu opartego na gęstości przestrzennej (HDBSCAN) bezpośrednio na tak szerokim zbiorze doprowadziłoby do zjawiska tzw. Klątwy Wymiarowości (Curse of Dimensionality) – w wysokich wymiarach metryki odległości (np. euklidesowa) tracą matematyczny sens, a wszystkie punkty wydają się być równie odległe od siebie.
- Zastosowano algorytm nieliniowej redukcji wymiarowości UMAP (Uniform Manifold Approximation and Projection).
- Dlaczego 5 wymiarów? Redukcja do 2-3 wymiarów prowadzi do zjawiska "crowdingu" (zbyt dużej utraty informacji i zlewania się różnych ataków w jedną plamę). Z kolei pozostawienie więcej niż 15-20 wymiarów drastycznie wydłuża czas obliczeń HDBSCAN i osłabia jego zdolność do wyłapywania szumu. Wartość <span style="color: red;"> n_components=10 </span> stanowi udowodniony empirycznie kompromis: zachowuje globalną strukturę topologiczną ataków (manifold), jednocześnie kompresując przestrzeń na tyle mocno, by algorytm gęstościowy mógł skutecznie wyrysować ostre granice klastrów i odizolować promieniowanie tła (Background Noise).

# Funkcje Pomocnicze

In [2]:
import numpy as np
import pandas as pd

def print_traffic_passport(df, title="PASZPORT RUCHU SIECIOWEGO"):
    def get_mode(x):
        m = pd.Series.mode(x)
        return m.iloc[0] if not m.empty else "N/A"

    # 1. Agregacja (wciąż na danych zalogarytmowanych, bo to uśrednia outliery!)
    report = df.groupby('Refined_Label').agg({
        'Dur': 'median', 
        'TotPkts': 'median', 
        'Bytes_per_Pkt': 'mean',
        'State': get_mode, 
        'Dport': get_mode,
        'srcUdata': lambda x: x.dropna().iloc[0] if not x.dropna().empty else "[Brak Payloadu]"
    }).reset_index()

    # 2. Liczność klas
    class_sizes = df['Refined_Label'].value_counts().to_dict()
    report['Liczność'] = report['Refined_Label'].map(class_sizes)
    report = report.sort_values(by='Liczność', ascending=False)

    # 3. Estetyczny nagłówek 
    print("\n" + "="*125)
    print(f"{title}")
    print("="*125)
    print(f"{'KLASA (Refined_Label)':<36} | {'LICZNOŚĆ':<9} | {'PORT':<6} | {'PKT':<6} | {'BpP':<8} | {'STATE':<6} | {'PRÓBKA PAYLOADU'}")
    print("-" * 125)

    # 4. Drukowanie wierszy z odzyskiwaniem prawdziwej fizyki (expm1)
    for _, row in report.iterrows():
        # Czyszczenie Payloadu (usuwanie enterów, żeby nie łamały tabeli)
        p_clean = str(row['srcUdata']).replace('\n', ' ').replace('\r', ' ')
        p_clean = p_clean[:35] + "..." if len(p_clean) > 35 else p_clean
        
        # ODKODOWANIE LOGARYTMU: np.expm1 odwraca np.log1p
        real_pkts = int(np.expm1(row['TotPkts'])) if row['TotPkts'] > 0 else 0
        real_bpp = float(np.expm1(row['Bytes_per_Pkt'])) if row['Bytes_per_Pkt'] > 0 else 0.0
        
        # CZYSZCZENIE PORTÓW: usunięcie miejsc po przecinku (25.0 -> 25)
        try:
            port_clean = str(int(float(row['Dport'])))
        except:
            port_clean = str(row['Dport'])
            
        # Formatowanie wiersza
        print(f"{str(row['Refined_Label']):<36} | {int(row['Liczność']):<9} | {port_clean:<6} | {real_pkts:<6} | {real_bpp:<8.1f} | {str(row['State']):<6} | {p_clean}")
        
    print("="*125)

 Definiujemy uniwersalną funkcję <span style="color: red;"> print_traffic_passport </span>, która działa jak narzędzie klasy DPI (Deep Packet Inspection). Pozwala ona na weryfikację fizycznego ładunku (payloadu) i atrybutów sieciowych dla każdej wykrytej grupy w dowolnym momencie trwania analizy. Zastosowanie zasady DRY (Don't Repeat Yourself) ułatwia późniejsze profilowanie zagrożeń.

# HDBSCAN | Pseudo-Labeling | Profilowanie 

In [3]:
import hdbscan
import time
import pandas as pd
import numpy as np

print("="*80)
print(">>> KROK 2: HDBSCAN - CONSTRAINED HYBRID LABELING <<<")
print("="*80)

# ==========================================
# 1. KLASTROWANIE HDBSCAN
# ==========================================
start_time = time.time()
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=100, 
    min_samples=15, 
    metric='euclidean', 
    cluster_selection_method='eom', 
    core_dist_n_jobs=1, 
    prediction_data=True
)
hdb_labels = clusterer.fit_predict(X_train_umap)
hdb_probs = clusterer.probabilities_ # Wymagane dla rdzeni (Exemplars)
print(f"[*] HDBSCAN zakończony w {time.time() - start_time:.2f} s.")

df_hdbscan = df_train_eng.copy() 
df_hdbscan['Cluster'] = hdb_labels
df_hdbscan['Cluster_Prob'] = hdb_probs


# ==========================================
# 2. DEFINICJA HEURYSTYKI BAZOWEJ
# ==========================================
SERVICE_MAP = {
    17: 'QOTD', 21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP', 53: 'DNS',
    80: 'HTTP', 111: 'RPCbind', 123: 'NTP', 135: 'SMB', 137: 'NetBIOS',
    139: 'SMB', 161: 'SNMP', 389: 'LDAP', 443: 'HTTPS', 445: 'SMB',
    500: 'ISAKMP_VPN', 1900: 'SSDP', 3306: 'MySQL', 3389: 'RDP',
    3702: 'WS-Discovery', 5060: 'SIP', 5070: 'SIP_Alt', 5353: 'mDNS',
    5683: 'CoAP', 8889: 'Web_Alt_Bin'
}

def safe_port_convert(port_val):
    try: return int(port_val, 16) if isinstance(port_val, str) and port_val.startswith('0x') else int(float(port_val))
    except: return -1

def get_label_for_row(row):
    port = safe_port_convert(row['Dport'])
    proto = str(row['Proto']).lower()
    payload = str(row.get('srcUdata', '')).lower()
    state = str(row.get('State', ''))
    
    # >>> Odwrócenie logarytmu (expm1), aby odzyskać surową fizykę! <<<
    pkts = int(np.expm1(float(row['TotPkts']))) if row['TotPkts'] > 0 else 0
    bpp = float(np.expm1(float(row['Bytes_per_Pkt']))) if row['Bytes_per_Pkt'] > 0 else 0
    dur = float(np.expm1(float(row['Dur']))) if row['Dur'] > 0 else 0
    
    service = SERVICE_MAP.get(port, f"{proto.upper()}_Port_{port}" if port != -1 else "GENERIC")
    
    if proto == 'icmp' or state == 'ECO': return "ICMP Fingerprinting (OS Detection)"
    if 'get /' in payload and port not in [80, 443, 8080]: return "HTTP Cross-Protocol Anomaly"
    if 'm-search' in payload: return "SSDP/UPnP Amplification Recon"
    
    if proto == 'tcp' and pkts <= 5 and bpp <= 65: return "Mass TCP Port Scanning (L4)"
    if dur > 60 and pkts < 20: return "Long-lived Session Anomaly"
    if pkts > 15 and dur > 1.0: return f"{service} Brute Force"
    if pkts <= 5: return f"{service} Enumeration"
    
    return f"{service} Anomaly"

# ==========================================
# 3. CONSTRAINED EXEMPLAR-BASED LABEL PROPAGATION
# ==========================================
print("[*] Rozpoczynanie hybrydowej propagacji etykiet (Constrained Pseudo-Labeling)...")

unique_clusters = set(hdb_labels) - {-1}
cluster_label_map = {}

for cluster_id in unique_clusters:
    # A. Izolacja klastra
    cluster_points = df_hdbscan[df_hdbscan['Cluster'] == cluster_id]
    
    # B. Wyodrębnienie rdzenia (Exemplars)
    top_quantile = cluster_points['Cluster_Prob'].quantile(0.85)
    exemplars = cluster_points[cluster_points['Cluster_Prob'] >= top_quantile]
    
    # C. Głosowanie w rdzeniu
    candidate_labels = exemplars.apply(get_label_for_row, axis=1)
    label_counts = candidate_labels.value_counts(normalize=True)
    
    winning_label = label_counts.index[0]
    purity = label_counts.iloc[0]
    
    # D. Weryfikacja czystości semantycznej klastra (>70% spójności)
    if purity >= 0.70:
        cluster_label_map[cluster_id] = winning_label
    else:
        cluster_label_map[cluster_id] = "MIXED_CLUSTER"


def apply_constrained_label(row):
    if row['Cluster'] == -1:
        return 'Background Noise'
        
    cluster_proposed_label = cluster_label_map.get(row['Cluster'], 'Unclassified')
    row_physical_label = get_label_for_row(row)
    
    if cluster_proposed_label == "MIXED_CLUSTER":
        return row_physical_label
        
    # BRAMKA SEMANTYCZNA: Zapobiega nadpisaniu portu fizycznego przez obcy klaster
    proposed_service = cluster_proposed_label.split()[0]
    physical_service = row_physical_label.split()[0]
    
    # Wygładzamy tylko te anomalie, które zgadzają się z główną osią fizyczną klastra
    if proposed_service == physical_service:
        return cluster_proposed_label
    else:
        return row_physical_label


df_hdbscan['Refined_Label'] = df_hdbscan.apply(apply_constrained_label, axis=1)
print(f"[*] Proces zakończony. Rozpropagowano etykiety z zachowaniem twardych ograniczeń portów.")



>>> KROK 2: HDBSCAN - CONSTRAINED HYBRID LABELING <<<
[*] HDBSCAN zakończony w 20.62 s.
[*] Rozpoczynanie hybrydowej propagacji etykiet (Constrained Pseudo-Labeling)...
[*] Proces zakończony. Rozpropagowano etykiety z zachowaniem twardych ograniczeń portów.


### Konfiguracja klastrowania i hybrydowa propagacja etykiet <span style="font-size: 18px;"> 

W tym etapie proces grupowania ruchu realizowany jest przy użyciu algorytmu HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise). Wybór tego algorytmu podyktowany jest jego zdolnością do wykrywania klastrów o dowolnym kształcie oraz natywnej izolacji szumu informacyjnego.
Kluczowe parametry konfiguracyjne:
- <span style="color: red;">min_cluster_size=100:</span> Przyjęto założenie, że statystycznie istotna kampania ataku w logach honeypota musi generować minimum 100 zdarzeń. Pozwala to na odseparowanie zorganizowanych działań od incydentalnych anomalii.
- <span style="color: red;"> min_samples=15: </span> Parametr ten wymusza wysoką gęstość klastrów, co ogranicza ryzyko łączenia odrębnych zachowań sieciowych w jedną strukturę (tzw. ochrona przed zjawiskiem chaining).
- <span style="color: red;"> cluster_selection_method='eom' </span> (Excess of Mass): Zapewnia stabilność klastrów poprzez wybór struktur o największej masie gęstości, co zapobiega nadmiernej fragmentacji dużych, jednorodnych kampanii skanowania.

Constrained Exemplar-based Labeling
Zaimplementowany został autorski, hybrydowy system etykietowania, składający się z trzech filarów bezpieczeństwa

- Exemplar Extraction (Wyodrębnienie Rdzenia): Zamiast patrzeć na cały (często zaszumiony na obrzeżach) klaster, algorytm izoluje 15% punktów o najwyższym prawdopodobieństwie przynależności (HDBSCAN probabilities_). Stanowią one sterylny, matematyczny archetyp ataku
- Cluster Purity Check (Kontrola Czystości): Algorytm sprawdza, czy rdzeń jest jednorodny. Jeśli wewnątrz rdzenia zgadza się mniej niż 70% etykiet fizycznych, klaster zostaje uznany za sztucznie sklejony przez UMAP i otrzymuje flagę MIXED_CLUSTER.
- Semantic Gate (Bramka Semantyczna): To najważniejszy element mechanizmu. Zanim wiersz odziedziczy etykietę od klastra, następuje twarda weryfikacja. Fizyczny port wiersza musi zgadzać się z usługą proponowaną przez klaster. Jeśli UMAP spróbuje narzucić etykietę "DNS" pakietowi lecącemu na port 80, Bramka Semantyczna to zablokuje i wymusi powrót do prawdy fizycznej (Row-by-Row fallback).</span>

# Walidacja | Deep Packet Inspection

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import umap.umap_ as umap
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
import os
import random

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

print("="*80)
print(">>> KROK 2.5: WSTĘPNA WALIDACJA MODELU (PRZED REKURENCJĄ) <<<")
print("="*80)

print_traffic_passport(df_hdbscan, title="PASZPORT RUCHU (PRZED REKURENCJNYM KLASTROWANIEM)")

# ==========================================
# 2. MASTER VALIDATION SUITE
# ==========================================

def run_full_validation_suite(df_analyzed, X_geom, clusterer_obj, label_col='Refined_Label', min_plot_records=10, sample_size=15000, output_dir='../wykresy'):
    print(f"\n##########################################################")
    print(f"###                 Walidacja modelu                   ###")
    print(f"##########################################################\n")
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 1. PRZYGOTOWANIE DANYCH (Zabezpieczenie Indeksów)
    df = df_analyzed.copy().reset_index(drop=True)
    df[label_col] = df[label_col].astype(str)
    
    # --- ODWROTNOŚĆ LOGARYTMÓW DO WYKRESÓW BOXPLOT (Critical Fix) ---
    for col in ['Dur', 'TotBytes']:
        if col in df.columns:
            df[f'{col}_real'] = np.expm1(df[col]).clip(lower=0)
    
    # 2. INTELIGENTNE GRUPOWANIE DŁUGIEGO OGONA (Tylko dla wykresów!)
    class_counts = df[label_col].value_counts()
    valid_classes = class_counts[class_counts >= min_plot_records].index
    
    df['Plot_Label'] = df[label_col].apply(lambda x: x if x in valid_classes else 'Minor Anomalies')
    df['Plot_Label'] = df['Plot_Label'].replace('Background Noise', 'Noise')

    # 3. KOLORYSTYKA
    unique_labels = sorted([l for l in df['Plot_Label'].unique() if l not in ['Noise', 'Minor Anomalies']])
    n_labels = len(unique_labels)
    palette_raw = sns.color_palette("husl", n_labels)
    random.seed(42)
    
    color_dict = dict(zip(unique_labels, palette_raw))
    color_dict['Noise'] = '#B0B0B0'
    color_dict['Minor Anomalies'] = '#000000'
    
    hue_order = unique_labels + ['Minor Anomalies', 'Noise']

    # --- [A] METRYKI ---
    print(">>> [A] OBLICZANIE METRYK WEWNĘTRZNYCH (HDBSCAN)...")
    labels_raw = df['Cluster'].values
    mask = labels_raw != -1
    X_clean = X_geom[mask]
    y_clean = labels_raw[mask]
    
    metrics_text = ""
    if len(set(y_clean)) < 2:
        metrics_text = "Za mało klastrów do oceny."
    else:
        if len(X_clean) > sample_size:
            idx = np.random.choice(len(X_clean), sample_size, replace=False)
            X_samp, y_samp = X_clean[idx], y_clean[idx]
        else:
            X_samp, y_samp = X_clean, y_clean
        
        sil = silhouette_score(X_samp, y_samp)
        db = davies_bouldin_score(X_samp, y_samp)
        metrics_text += f"Silhouette Score: {sil:.4f}\nDavies-Bouldin:   {db:.4f}\n"
        
        try:
            from hdbscan.validity import validity_index
            X_samp_64 = X_samp.astype(np.float64) 
            dbc_score = validity_index(X_samp_64, y_samp, metric='euclidean')
            metrics_text += f"DBCV (Index):       {dbc_score:.4f}\n"
        except Exception as e:
            metrics_text += f"DBCV: Błąd obliczeń ({str(e)})\n"
        print(metrics_text)

    # --- [B] HISTOGRAM PEWNOŚCI ---
    print(">>> [B] GENEROWANIE HISTOGRAMU PEWNOŚCI...")
    probs = clusterer_obj.probabilities_
    mask_noise = labels_raw == -1
    plt.figure(figsize=(10, 5))
    plt.hist(probs[~mask_noise], bins=40, color='forestgreen', alpha=0.7, label='Zdefiniowane Klastry', density=True)
    plt.hist(probs[mask_noise], bins=40, color='firebrick', alpha=0.7, label='Szum HDBSCAN', density=True)
    plt.title("Jakość Separacji: Histogram Pewności Modelu", fontsize=12)
    plt.legend()
    plt.savefig(os.path.join(output_dir, '1_probability_histogram.png'), bbox_inches='tight')
    plt.close()

    # --- [C] MAPA 2D (UMAP) ---
    print(">>> [C] GENEROWANIE MAPY UMAP 2D...")
    reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42, n_jobs=1)
    if len(X_geom) > sample_size:
        plot_idx = np.random.choice(len(X_geom), sample_size, replace=False)
        X_plot = X_geom[plot_idx]
        y_plot = df.iloc[plot_idx]['Plot_Label']
    else:
        X_plot = X_geom
        y_plot = df['Plot_Label']
        
    embedding_2d = reducer_2d.fit_transform(X_plot)
    plt.figure(figsize=(14, 9))
    sns.scatterplot(x=embedding_2d[:, 0], y=embedding_2d[:, 1], hue=y_plot, hue_order=hue_order, palette=color_dict, s=15, alpha=0.8)
    plt.title("Topologia Ataków (UMAP 2D) - Semantic Labels", fontsize=14)
    plt.legend(bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0., fontsize='small', title="Klasa Ataku")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '2_umap_topology.png'), bbox_inches='tight')
    plt.close()

    # --- [D] DRZEWO SKONDENSOWANE (HDBSCAN) ---
    print(">>> [D] GENEROWANIE DRZEWA SKONDENSOWANEGO HDBSCAN...")
    plt.figure(figsize=(12, 8))
    
    try:
        clusterer.condensed_tree_.plot(
            select_clusters=True, 
            selection_palette=sns.color_palette('tab20', len(set(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0))
        )
        plt.title('Hierarchia gęstości HDBSCAN (Condensed Tree)', fontsize=15)
        plt.ylabel('Lambda (Miara gęstości / 1/distance)', fontsize=12)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '3_hdbscan_condensed_tree.png'), bbox_inches='tight')
        plt.close()
        print("[+] SUKCES: Wykres Condensed Tree został zapisany!")
    except Exception as e:
        print(f"[-] BŁĄD przy generowaniu Condensed Tree: {e}. Sprawdź nazwę zmiennej modelu HDBSCAN.")
    
    # --- [E] TIMELINE ---
    print(">>> [D] GENEROWANIE TIMELINE...")
    col_time = None
    for c in ['StartTime', 'stime', 'ts', 'Time']:
        if c in df.columns:
            col_time = c
            break
            
    if col_time:
        df_time = df.copy()
        df_time['dt'] = pd.to_datetime(df_time[col_time], errors='coerce')
        df_time = df_time.dropna(subset=['dt'])
        df_time_shuffled = df_time.sample(frac=1, random_state=42).reset_index(drop=True)
        
        plt.figure(figsize=(16, 9))
        sns.scatterplot(data=df_time_shuffled, x='dt', y='Plot_Label', hue='Plot_Label', hue_order=hue_order, palette=color_dict, s=30, legend=False, alpha=0.9)
        plt.title("Dynamika Zagrożeń w Czasie", fontsize=14)
        plt.grid(True, alpha=0.4, linestyle='--')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '3_timeline_analysis.png'), bbox_inches='tight')
        plt.close()
    else:
        print("[Warn] Brak kolumny czasu.")

    # --- [F] SYGNATURY WIZUALNE (BOXPLOTS) ---
    print(">>> [E] GENEROWANIE SYGNATUR...")
    fig, axes = plt.subplots(1, 2, figsize=(18, 8)) 
    labels_no_noise = [l for l in hue_order if l != 'Noise']
    
    # UŻYWAMY ODKODOWANYCH WARTOSCI (_real) i logarytmicznej osi Y
    if 'Dur_real' in df.columns:
        sns.boxplot(data=df[df['Plot_Label'] != 'Noise'], x='Plot_Label', y='Dur_real', hue='Plot_Label', legend=False, ax=axes[0], palette=color_dict, showfliers=False, order=labels_no_noise)
        axes[0].set_title("Sygnatura Czasowa (Duration) [Sekundy]", fontsize=12)
        axes[0].tick_params(axis='x', rotation=90)
        axes[0].set_yscale('log')
    
    if 'TotBytes_real' in df.columns:
        sns.boxplot(data=df[df['Plot_Label'] != 'Noise'], x='Plot_Label', y='TotBytes_real', hue='Plot_Label', legend=False, ax=axes[1], palette=color_dict, showfliers=False, order=labels_no_noise)
        axes[1].set_title("Sygnatura Wolumetryczna (Bytes) [Bajty]", fontsize=12)
        axes[1].tick_params(axis='x', rotation=90)
        axes[1].set_yscale('log')
        
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '4_signatures_boxplots.png'), bbox_inches='tight')
    plt.close()

    # --- [G] HEATMAPA (FINGERPRINTING) ---
    print(">>> [F] GENEROWANIE HEATMAPY STANÓW TCP...")
    fingerprint = pd.crosstab(df['Plot_Label'], df['State'], normalize='index')
    top_attacks = df['Plot_Label'].value_counts().index
    fingerprint = fingerprint.reindex(top_attacks)
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(fingerprint, cmap='magma_r', linewidths=.5, linecolor='lightgray', annot=False)
    plt.title("Atak vs Stan Połączenia (Fingerprinting Behawioralny)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '5_fingerprint_heatmap.png'), bbox_inches='tight')
    plt.close()

    # --- [H] KORELACJE ---
    print(">>> [G] OBLICZANIE MACIERZY KORELACJI...")
    cols_corr = ['Dur', 'TotPkts', 'TotBytes', 'Bytes_per_Pkt', 'Load', 'Rate', 'SrcPkts', 'DstPkts', 'pLoss']
    cols_exist = [c for c in cols_corr if c in df.columns]
    
    numeric_df = df[cols_exist].copy()
    corr = numeric_df.corr(method='spearman')
    mask_tri = np.triu(np.ones_like(corr, dtype=bool))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, mask=mask_tri, cmap='coolwarm', vmax=1.0, vmin=-1.0, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", cbar_kws={"shrink": .8})
    plt.title("Macierz Korelacji Cech Sieciowych", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '6_correlation_matrix.png'), bbox_inches='tight')
    plt.close()
    
    print(f"\n[SUKCES] Komplet wszystkich 7 zestawów wizualizacji zapisany w folderze '{output_dir}'.")

# URUCHOMIENIE WALIDACJI (Tu wstawiasz gotowy zbiór, np. przed Długim Ogonem)
run_full_validation_suite(df_hdbscan, X_train_umap, clusterer, label_col='Refined_Label')

>>> KROK 2.5: WSTĘPNA WALIDACJA MODELU (PRZED REKURENCJĄ) <<<

PASZPORT RUCHU (PRZED REKURENCJNYM KLASTROWANIEM)
KLASA (Refined_Label)                | LICZNOŚĆ  | PORT   | PKT    | BpP      | STATE  | PRÓBKA PAYLOADU
-----------------------------------------------------------------------------------------------------------------------------
Mass TCP Port Scanning (L4)          | 33790     | 25     | 3      | 57.5     | SR_RA  | s[6]=......
Background Noise                     | 28226     | 25     | 2      | 62.0     | S_RA   | s[54]=................................
ICMP Fingerprinting (OS Detection)   | 1341      | 0      | 2      | 89.5     | ECO    | s[64]=......0i.yD.%U..................
SIP Enumeration                      | 476       | 5060   | 1      | 424.5    | INT    | s[413]=OPTIONS sip:100@167.71.64.32...
ISAKMP_VPN Brute Force               | 280       | 500    | 278    | 434.0    | REQ    | s[480]=&...=.EK........! "............
Web_Alt_Bin Brute Force              | 233 

<span style="font-size: 18px;"> Automatyczna walidacja oceniająca jakość klastrowania. Generujemy zestaw metryk wewnętrznych (Silhouette, DBCV, Davies-Bouldin) oraz 7 wizualizacji (m.in. mapy topologii UMAP, sygnatury wolumetryczne w postaci wykresów pudełkowych, heatmapy stanów TCP). Pozwala to na fizyczną weryfikację, czy wydzielone klastry faktycznie reprezentują odmienne typy zachowań sieciowych adwersarzy. </span>

# Recursive Noise Mining | Final Correction

In [5]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import umap.umap_ as umap
import hdbscan
import os

print("="*80)
print(">>> KROK 3: REKURENCYJNE KLASTROWANIE SZUMU (NOISE MINING) <<<")
print("="*80)

# 1. Wyciągamy szum z poprzedniego kroku
df_noise = df_hdbscan[df_hdbscan['Refined_Label'] == 'Background Noise'].copy()
df_final = df_hdbscan.copy() # Baza, na której będziemy pracować do samego końca

# BEZPIECZNIK: Jeśli nie ma szumu, pomijamy ten krok
if len(df_noise) < 15: 
    print(f"[*] Zbyt mała ilość szumu do klastrowania ({len(df_noise)} rekordów). Pomijam Noise Mining.")
else:
    print(f"[*] Wyizolowano {len(df_noise)} rekordów szumu. Eksploracja...")

    # Przygotowanie i skalowanie szumu (Lokalna struktura)
    features_to_use = ['Dur', 'TotPkts', 'TotBytes', 'SrcPkts', 'DstPkts']
    if 'Bytes_per_Pkt' in df_noise.columns: features_to_use.append('Bytes_per_Pkt')
    
    X_noise_scaled = StandardScaler().fit_transform(df_noise[features_to_use].fillna(0).values)

    reducer_noise = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.01, random_state=42, n_jobs=1)
    X_noise_umap = reducer_noise.fit_transform(X_noise_scaled)

    clusterer_noise = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean', cluster_selection_method='eom')
    df_noise['Sub_Cluster'] = clusterer_noise.fit_predict(X_noise_umap)

    unique_micro_clusters = df_noise['Sub_Cluster'].nunique() - 1
    print(f"[*] Odkryto {unique_micro_clusters} nowych mikrogrup w obrębie szumu.")

    def map_noise_micro_clusters(row):
        if row['Sub_Cluster'] == -1: 
            return "Background Noise"
        # Jeśli HDBSCAN znalazł strukturę, ufamy naszej głównej funkcji!
        return get_label_for_row(row) 

    df_noise['Final_Refined_Label'] = df_noise.apply(map_noise_micro_clusters, axis=1)

    # Scalenie z głównym zbiorem
    df_final.loc[df_noise.index, 'Refined_Label'] = df_noise['Final_Refined_Label']

# ==========================================
# 2. OSTATECZNA KOREKTA SEMANTYCZNA
# ==========================================

def semantic_ground_truth_correction(row):
    label = str(row['Refined_Label'])
    port = safe_port_convert(row['Dport'])
    payload = str(row.get('srcUdata', '')).lower()
    
    if label == 'HTTP Enumeration' and port == 111 and 'objectclass' in payload: return 'LDAP Enumeration'
    elif label == 'SMTP Enumeration' and port == 17: return 'QOTD/Legacy Service Scan'
    elif label == 'TCP_Port_53413 Scanning' and 'get / http' in payload: return 'HTTP Cross-Protocol Anomaly'
    elif label == 'TCP_Port_6144 Scanning': return 'Generic UDP Scan'
    elif label == 'HTTP Anomaly' and '[Brak Payloadu]' in str(row.get('srcUdata', '')): return 'Mass TCP Port Scanning (L4)'
    return label

df_final['Refined_Label'] = df_final.apply(semantic_ground_truth_correction, axis=1)

# ==========================================
# 3. OCZYSZCZANIE ZBIORU POD ML (Agregacja Długiego Ogona)
# ==========================================
print("\n[*] Przygotowanie finalnego zbioru pod Machine Learning...")

THRESHOLD = 30
class_counts = df_final['Refined_Label'].value_counts()
valid_classes = class_counts[class_counts >= THRESHOLD].index

df_final.loc[~df_final['Refined_Label'].isin(valid_classes), 'Refined_Label'] = 'Rare / Minor Anomalies'

print(f"[*] Skonwertowano małe klastry (poniżej {THRESHOLD} próbek) do grupy 'Rare / Minor Anomalies'.")
print(f"[*] Finalna liczba rekordów gotowych do treningu ML: {len(df_final)}")

# Wyświetlenie ostatecznego rozkładu
print("\nRozkład klas po agregacji Długiego Ogona:")
print(df_final['Refined_Label'].value_counts().head(15))

# ==========================================
# 4. WYDRUK I EKSPORT
# ==========================================
print_traffic_passport(df_final, title="OSTATECZNY PASZPORT RUCHU (GOTOWY DO ML)")

print("\n[*] Finalizacja zbioru danych...")

df_ml_final = df_final[df_final['Refined_Label'] != 'Background Noise'].copy()

print(f"[*] Usunięto klasę 'Background Noise' ({len(df_final) - len(df_ml_final)} rekordów).")
print(f"[*] Klasa 'Rare / Minor Anomalies' została zachowana jako wzorzec rzadkich incydentów.")

# Eksport
output_dir_final = '../dane/gotowe_ml/'
if not os.path.exists(output_dir_final): 
    os.makedirs(output_dir_final)

output_file = output_dir_final + 'honeypot_ml_ready.csv'
df_ml_final.to_csv(output_file, index=False)

print(f"\n[SUKCES] Zapisano {len(df_ml_final)} wierszy.")
print(f"[+] Plik sterylny gotowy do modeli ML: {output_file}")

>>> KROK 3: REKURENCYJNE KLASTROWANIE SZUMU (NOISE MINING) <<<
[*] Wyizolowano 28226 rekordów szumu. Eksploracja...
[*] Odkryto 297 nowych mikrogrup w obrębie szumu.

[*] Przygotowanie finalnego zbioru pod Machine Learning...
[*] Skonwertowano małe klastry (poniżej 30 próbek) do grupy 'Rare / Minor Anomalies'.
[*] Finalna liczba rekordów gotowych do treningu ML: 65765

Rozkład klas po agregacji Długiego Ogona:
Refined_Label
Mass TCP Port Scanning (L4)           58834
ICMP Fingerprinting (OS Detection)     3070
Rare / Minor Anomalies                  915
SIP Enumeration                         500
Long-lived Session Anomaly              320
NTP Enumeration                         292
ISAKMP_VPN Brute Force                  280
Web_Alt_Bin Brute Force                 274
Background Noise                        193
SSDP/UPnP Amplification Recon           192
LDAP Enumeration                        187
DNS Enumeration                         160
HTTP Brute Force                         77


### Eksploracja szumu (Recursive Noise Mining)<span style="font-size: 18px;"> 

Szum odrzucony w pierwszym przebiegu HDBSCAN (Label -1) może wciąż zawierać celowe, ale rzadkie ataki (np. punktowe uderzenia w nietypowe porty UDP). Aby wyizolować te mikro-anomalie z tła internetowego, zastosowano klastrowanie rekurencyjne o podwyższonej czułości:
- UMAP (min_dist=0.01): Parametr kompresji przestrzennej został ekstremalnie obniżony. Zmusza to algorytm do bardzo ciasnego pakowania identycznych pakietów, uwydatniając małe grupy.
- HDBSCAN (min_cluster_size=15, min_samples=5): Drastyczne obniżenie progów względem głównego modelu. Tym razem szukamy ukrytych mikrogrup (minimum 15 rekordów) i pozwalamy na klastrowanie przy znacznie mniejszej gęstości otoczenia.

Znalezione w ten sposób mikro-anomalie (m.in. SSDP Amplification czy skany DNS) są ponownie mapowane, co ostatecznie domyka proces tworzenia czystego zbioru Ground Truth.<span>
